In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

zip_path = "/content/drive/MyDrive/CropIQ/CropIQ.zip"
extract_path = "/content/full_dataset"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

print("Dataset extracted.")
train_root = '/content/full_dataset/Fruit 360/Training'
val_root   = '/content/full_dataset/Fruit 360/Validation'
test_root  = '/content/full_dataset/Fruit 360/Test'
print("Sample folders:", os.listdir(train_root)[:10])

In [ ]:
import shutil, re
from collections import Counter

# All parent classes from Fruit-360 dataset
FRUIT360_PARENTS = [
    "Apple", "Apricot", "Avocado", "Banana", "Blackberry",
    "Blueberry", "Cactus fruit", "Cantaloupe", "Carambola", "Cauliflower",
    "Cherry", "Clementine", "Coconut", "Corn", "Cucumber",
    "Dates", "Eggplant", "Fig", "Ginger", "Grape",
    "Grapefruit", "Guava", "Hazelnut", "Huckleberry", "Kaki",
    "Kiwi", "Kumquat", "Lemon", "Limes", "Lychee",
    "Mango", "Maracuja", "Melon", "Mulberry", "Nectarine",
    "Nut", "Onion", "Orange", "Papaya", "Passion fruit",
    "Peach", "Pear", "Pepper", "Physalis", "Pineapple",
    "Pistachio", "Plum", "Pomegranate", "Pomelo", "Potato",
    "Pumpkin", "Quince", "Radish", "Raspberry", "Redcurrant",
    "Salak", "Strawberry", "Tomato", "Walnut", "Watermelon",
    "Zucchini"
]

# Our 13 target classes + background
TARGET_CLASSES = [
    "Apple", "Banana", "Cabbage", "Carrot",
    "Cucumber", "Eggplant", "Grape", "Onion",
    "Orange", "Papaya", "Pepper", "Strawberry",
    "Tomato"
]

# Build mapping: folder prefix -> target class (case-insensitive)
def build_folder_map(src_root):
    folders = [f for f in os.listdir(src_root) if os.path.isdir(os.path.join(src_root, f))]
    mapping = {}
    for folder in folders:
        # Extract parent class from folder name
        match = re.match(r'^([a-zA-Z]+(?:\s+[a-zA-Z]+)*)', folder)
        if not match:
            continue
        prefix = match.group(1)
        # Exact match to target classes
        for tc in TARGET_CLASSES:
            if tc.lower() == prefix.lower():
                mapping[folder] = tc
                break
        # Fallback: match against all Fruit-360 parents
        if folder not in mapping:
            for parent in FRUIT360_PARENTS:
                if parent.lower() == prefix.lower():
                    # Map to closest target class
                    for tc in TARGET_CLASSES:
                        if tc.lower() in parent.lower() or parent.lower() in tc.lower():
                            mapping[folder] = tc
                            break
                    if folder in mapping:
                        break
    return mapping

folder_to_class = build_folder_map(train_root)
print(f"Mapped {len(folder_to_class)} folders to {len(TARGET_CLASSES)} classes")

dst_root = '/content/merged_dataset'
counts = {split: Counter() for split in ['Training', 'Validation', 'Test']}

for split, src_split in [('Training', train_root), ('Validation', val_root), ('Test', test_root)]:
    for cls in TARGET_CLASSES:
        os.makedirs(os.path.join(dst_root, split, cls), exist_ok=True)
    for folder_name in os.listdir(src_split):
        src_folder = os.path.join(src_split, folder_name)
        if not os.path.isdir(src_folder):
            continue
        parent = folder_to_class.get(folder_name)
        if parent is None:
            continue
        dst_folder = os.path.join(dst_root, split, parent)
        for fname in os.listdir(src_folder):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.gif')):
                shutil.copy2(os.path.join(src_folder, fname), dst_folder)
                counts[split][parent] += 1
        print(f"{folder_name} -> {parent}")

print("\n=== Final class counts ===")
for cls in TARGET_CLASSES:
    print(f"{cls:15s} train={counts['Training'][cls]:5d}  val={counts['Validation'][cls]:5d}  test={counts['Test'][cls]:5d}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import ssl, urllib.request, zipfile, os, random, shutil
from tqdm.notebook import tqdm

ADD_BACKGROUND_CLASS = True

bg_url = "https://images.cocodataset.org/zips/unlabeled2017.zip"
bg_dir = "/content/drive/MyDrive/CropIQ/coco_unlabeled"

if ADD_BACKGROUND_CLASS and not os.path.exists(bg_dir):
    print("Downloading COCO unlabeled2017.zip...")
    ctx = ssl._create_unverified_context()
    bg_zip = "/content/coco_unlabeled.zip"
    req = urllib.request.Request(bg_url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, context=ctx) as r:
        total = int(r.headers.get('Content-Length', 0))
        with open(bg_zip, 'wb') as f, tqdm(
            total=total, unit='B', unit_scale=True, unit_divisor=1024,
            desc="Downloading"
        ) as pbar:
            while True:
                chunk = r.read(8192)
                if not chunk: break
                f.write(chunk)
                pbar.update(len(chunk))

    print("Extracting first 600 images to Drive...")
    with zipfile.ZipFile(bg_zip, 'r') as z:
        members = z.namelist()[:600]
        for m in tqdm(members, desc="Extracting"):
            z.extract(m, bg_dir)
    os.remove(bg_zip)
    print(f"Saved 600 COCO images to {bg_dir}")

if ADD_BACKGROUND_CLASS:
    all_bg = []
    for root, dirs, files in os.walk(bg_dir):
        for f in files:
            if f.endswith('.jpg'):
                all_bg.append(os.path.join(root, f))
    random.shuffle(all_bg)

    train_bg = os.path.join(dst_root, 'Training', 'Background')
    val_bg   = os.path.join(dst_root, 'Validation', 'Background')
    test_bg  = os.path.join(dst_root, 'Test', 'Background')
    os.makedirs(train_bg, exist_ok=True)
    os.makedirs(val_bg, exist_ok=True)
    os.makedirs(test_bg, exist_ok=True)

    n_train = min(400, len(all_bg))
    n_val   = min(100, len(all_bg) - n_train)
    n_test  = min(50, len(all_bg) - n_train - n_val)

    for f in all_bg[:n_train]: shutil.copy2(f, train_bg)
    for f in all_bg[n_train:n_train + n_val]: shutil.copy2(f, val_bg)
    for f in all_bg[n_train + n_val:n_train + n_val + n_test]: shutil.copy2(f, test_bg)

    print("Background class added.")

In [ ]:
available_classes = [
    "Apple", "Banana", "Cabbage", "Carrot",
    "Cucumber", "Eggplant", "Grape", "Onion",
    "Orange", "Papaya", "Pepper", "Strawberry",
    "Tomato"
]

ADD_BACKGROUND_CLASS = True
final_classes = available_classes + (["Background"] if ADD_BACKGROUND_CLASS else [])
print("Training classes:", final_classes)

# Model 1: MobileNetV2

In [ ]:
import cv2, numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# GPU-compatible background replacement (runs inside model, not on CPU generator)
def replace_background_tf(images):
    """Replace white/near-white background with random color. images: [B,H,W,3] float32 [0,255]."""
    gray = tf.image.rgb_to_grayscale(images)
    mask = tf.cast(gray < 240, tf.float32)
    # Smooth mask to avoid hard edges
    mask = tf.nn.avg_pool2d(mask, 7, 1, 'SAME')
    bg = tf.random.uniform((tf.shape(images)[0], 1, 1, 3), 30, 225, dtype=tf.float32)
    return images * mask + bg * (1.0 - mask)

# Fast augmentations (C-level ops in ImageDataGenerator are fast)
train_datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.25,
    brightness_range=[0.7, 1.3],
    horizontal_flip=True,
    fill_mode='reflect'
)
# Validation: NO augmentation, NO rescale - raw [0,255] images
val_datagen = ImageDataGenerator()

train_generator = train_datagen.flow_from_directory(
    os.path.join(dst_root, 'Training'),
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=final_classes,
    shuffle=True, seed=42
)
val_generator = val_datagen.flow_from_directory(
    os.path.join(dst_root, 'Validation'),
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=final_classes,
    shuffle=False
)

print(f"Training samples: {train_generator.samples}, Validation: {val_generator.samples}")
print("Class indices:", train_generator.class_indices)

# Class weights for imbalanced data
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight(
    'balanced',
    classes=np.arange(len(final_classes)),
    y=train_generator.classes
)
class_weight_dict = {i: w for i, w in enumerate(class_weights)}
print("Class weights:", class_weight_dict)

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization, Lambda
from tensorflow.keras.applications import mobilenet_v2

num_classes = len(final_classes)

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = False

model_mobilenet = Sequential([
    Lambda(replace_background_tf, input_shape=(224,224,3), name='bg_replacement'),
    Lambda(mobilenet_v2.preprocess_input, name='mobilenet_preprocess'),
    base_model,
    GlobalAveragePooling2D(),
    BatchNormalization(),
    Dense(512, activation='relu'),
    BatchNormalization(),
    Dropout(0.4),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax', dtype='float32')
])

model_mobilenet.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
)
model_mobilenet.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import numpy as np
import math

EPOCHS_HEAD = 20
TOTAL_STEPS_HEAD = EPOCHS_HEAD * (train_generator.samples // BATCH_SIZE)
WARMUP_STEPS = TOTAL_STEPS_HEAD // 10

lr_schedule_head = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3,
    decay_steps=TOTAL_STEPS_HEAD - WARMUP_STEPS,
    alpha=1e-2
)

def warmup_cosine(step, warmup_steps, total_steps, base_lr=1e-3, min_lr=1e-5):
    step = tf.cast(step, tf.float32)
    warmup_steps = tf.cast(warmup_steps, tf.float32)
    total_steps = tf.cast(total_steps, tf.float32)
    warmup_lr = base_lr * (step / warmup_steps)
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    progress = tf.clip_by_value(progress, 0.0, 1.0)
    cosine_lr = min_lr + 0.5 * (base_lr - min_lr) * (1 + tf.cos(np.pi * progress))
    return tf.where(step < warmup_steps, warmup_lr, cosine_lr)

class WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, warmup_steps, total_steps, base_lr=1e-3, min_lr=1e-5):
        super().__init__()
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.base_lr = base_lr
        self.min_lr = min_lr
    def __call__(self, step):
        return warmup_cosine(step, self.warmup_steps, self.total_steps, self.base_lr, self.min_lr)
    def get_config(self):
        return {'warmup_steps': self.warmup_steps, 'total_steps': self.total_steps,
                'base_lr': self.base_lr, 'min_lr': self.min_lr}

# Phase 1: Train head only
lr_sched_head = WarmupCosineDecay(WARMUP_STEPS, TOTAL_STEPS_HEAD, 1e-3, 1e-5)
model_mobilenet.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=lr_sched_head, weight_decay=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
)

early1 = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, min_delta=1e-4)
checkpoint1 = ModelCheckpoint('/content/drive/MyDrive/CropIQ/best_mobilenet.keras', save_best_only=True)
reduce_lr1 = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)

print("=== Phase 1: Training Head ===")
history_mobilenet = model_mobilenet.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_HEAD,
    callbacks=[early1, checkpoint1, reduce_lr1],
    class_weight=class_weight_dict,
    verbose=1,
    workers=4, max_queue_size=20
)

# Phase 2: Fine-tune last 30 layers
base_model.trainable = True
# Freeze BatchNorm running stats on unfrozen layers
for layer in base_model.layers[-30:]:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
for layer in base_model.layers[:-30]:
    layer.trainable = False

EPOCHS_FINE = 20
TOTAL_STEPS_FINE = EPOCHS_FINE * (train_generator.samples // BATCH_SIZE)
lr_sched_fine = WarmupCosineDecay(TOTAL_STEPS_FINE // 10, TOTAL_STEPS_FINE, 5e-5, 1e-6)

model_mobilenet.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=lr_sched_fine, weight_decay=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
)

early2 = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, min_delta=1e-4)
checkpoint2 = ModelCheckpoint('/content/drive/MyDrive/CropIQ/best_mobilenet_finetuned.keras', save_best_only=True)
reduce_lr2 = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)

print("=== Phase 2: Fine-tuning ===")
history_mobilenet_fine = model_mobilenet.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_FINE,
    callbacks=[early2, checkpoint2, reduce_lr2],
    class_weight=class_weight_dict,
    verbose=1,
    workers=4, max_queue_size=20
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history_mobilenet.history['accuracy'], label='train_acc')
plt.plot(history_mobilenet.history['val_accuracy'], label='val_acc')
plt.title('MobileNetV2 Phase 1 Accuracy'); plt.legend()

plt.subplot(1,2,2)
plt.plot(history_mobilenet.history['loss'], label='train_loss')
plt.plot(history_mobilenet.history['val_loss'], label='val_loss')
plt.title('MobileNetV2 Phase 1 Loss'); plt.legend()
plt.show()

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history_mobilenet_fine.history['accuracy'], label='train_acc')
plt.plot(history_mobilenet_fine.history['val_accuracy'], label='val_acc')
plt.title('MobileNetV2 Fine-tune Accuracy'); plt.legend()

plt.subplot(1,2,2)
plt.plot(history_mobilenet_fine.history['loss'], label='train_loss')
plt.plot(history_mobilenet_fine.history['val_loss'], label='val_loss')
plt.title('MobileNetV2 Fine-tune Loss'); plt.legend()
plt.show()

# Model 2: EfficientNetB0

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization, Lambda
from tensorflow.keras.applications import efficientnet

base_model2 = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model2.trainable = False

model_efficientnet = Sequential([
    Lambda(replace_background_tf, input_shape=(224,224,3), name='bg_replacement'),
    Lambda(efficientnet.preprocess_input, name='efficientnet_preprocess'),
    base_model2,
    GlobalAveragePooling2D(),
    BatchNormalization(),
    Dense(512, activation='relu'),
    BatchNormalization(),
    Dropout(0.4),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(len(final_classes), activation='softmax', dtype='float32')
])

model_efficientnet.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
)
model_efficientnet.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import math

# Cosine decay with warmup schedule
class WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, warmup_steps, total_steps, base_lr, min_lr=1e-6):
        super().__init__()
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.base_lr = base_lr
        self.min_lr = min_lr
    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_lr = self.base_lr * (step / self.warmup_steps)
        progress = (step - self.warmup_steps) / (self.total_steps - self.warmup_steps)
        progress = tf.clip_by_value(progress, 0.0, 1.0)
        cosine_lr = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (1 + tf.cos(math.pi * progress))
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)

steps_per_epoch = train_generator.samples // BATCH_SIZE
EPOCHS_HEAD = 20
EPOCHS_FINE = 25
total_steps_head = steps_per_epoch * EPOCHS_HEAD
total_steps_fine = steps_per_epoch * EPOCHS_FINE
warmup_steps = steps_per_epoch * 2

lr_schedule_head = WarmupCosineDecay(warmup_steps, total_steps_head, base_lr=1e-3, min_lr=1e-5)

model_efficientnet.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=lr_schedule_head, weight_decay=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
)

early_head = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, min_delta=1e-4)
checkpoint_head = ModelCheckpoint('/content/drive/MyDrive/CropIQ/best_efficientnet.keras', save_best_only=True)
reduce_lr_head = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)

print("=== Phase 1: Training Head ===")
history_efficientnet = model_efficientnet.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_HEAD,
    callbacks=[early_head, checkpoint_head, reduce_lr_head],
    class_weight=class_weight_dict,
    verbose=1,
    workers=4, max_queue_size=20
)

In [ ]:
# Phase 2: Progressive fine-tuning
print("\n=== Phase 2: Progressive Fine-tuning ===")
val_generator.reset()
loss_p1, acc_p1 = model_efficientnet.evaluate(val_generator, verbose=1)
print(f"Phase 1 - Val Acc: {acc_p1:.4f}  Loss: {loss_p1:.4f}")

# Unfreeze progressively: start with last 30 layers
base_model2.trainable = True
for layer in base_model2.layers[:-30]:
    layer.trainable = False

# IMPORTANT: Freeze BatchNorm running stats during fine-tuning
# EfficientNet is BN-heavy; small batch sizes corrupt running stats
for layer in base_model2.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

lr_schedule_fine = WarmupCosineDecay(warmup_steps, total_steps_fine, base_lr=5e-5, min_lr=1e-7)

model_efficientnet.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=lr_schedule_fine, weight_decay=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
)

early_fine = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, min_delta=1e-4)
checkpoint_fine = ModelCheckpoint('/content/drive/MyDrive/CropIQ/best_efficientnet_finetuned.keras', save_best_only=True)
reduce_lr_fine = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7, verbose=1)

history_efficientnet_fine = model_efficientnet.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_FINE,
    callbacks=[early_fine, checkpoint_fine, reduce_lr_fine],
    class_weight=class_weight_dict,
    verbose=1,
    workers=4, max_queue_size=20
)

In [ ]:
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history_efficientnet.history['accuracy'], label='train_acc')
plt.plot(history_efficientnet.history['val_accuracy'], label='val_acc')
plt.title('EfficientNet Phase 1 Accuracy'); plt.legend()

plt.subplot(1,2,2)
plt.plot(history_efficientnet.history['loss'], label='train_loss')
plt.plot(history_efficientnet.history['val_loss'], label='val_loss')
plt.title('EfficientNet Phase 1 Loss'); plt.legend()
plt.show()

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history_efficientnet_fine.history['accuracy'], label='train_acc')
plt.plot(history_efficientnet_fine.history['val_accuracy'], label='val_acc')
plt.title('EfficientNet Fine-tune Accuracy'); plt.legend()

plt.subplot(1,2,2)
plt.plot(history_efficientnet_fine.history['loss'], label='train_loss')
plt.plot(history_efficientnet_fine.history['val_loss'], label='val_loss')
plt.title('EfficientNet Fine-tune Loss'); plt.legend()
plt.show()

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# No rescale - models handle preprocessing internally via Lambda layers
test_datagen = ImageDataGenerator()

test_generator = test_datagen.flow_from_directory(
    os.path.join(dst_root, 'Test'),
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=final_classes,
    shuffle=False
)

for name, mod in [('MobileNetV2', model_mobilenet), ('EfficientNetB0', model_efficientnet)]:
    test_generator.reset()
    test_loss, test_acc = mod.evaluate(test_generator, verbose=1)
    print(f"{name} - Test Accuracy: {test_acc:.4f}  |  Loss: {test_loss:.4f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import numpy as np

for name, mod in [('MobileNetV2', model_mobilenet), ('EfficientNetB0', model_efficientnet)]:
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    test_generator.reset()
    y_true = test_generator.classes
    y_pred_prob = mod.predict(test_generator, verbose=1)
    y_pred = np.argmax(y_pred_prob, axis=1)

    class_labels = list(test_generator.class_indices.keys())
    print(f"\nClassification Report:\n{classification_report(y_true, y_pred, target_names=class_labels)}")

    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
    fig, ax = plt.subplots(figsize=(14, 12))
    disp.plot(ax=ax, xticks_rotation=45, cmap='Blues', values_format='d')
    plt.title(f'{name} - Confusion Matrix')
    plt.show()

    class_acc = cm.diagonal() / cm.sum(axis=1)
    for i, (label, acc) in enumerate(zip(class_labels, class_acc)):
        print(f"  {label:20s}: {acc:.2%} ({cm.diagonal()[i]}/{cm.sum(axis=1)[i]})")
    print(f"  {'Overall':20s}: {np.sum(cm.diagonal()) / np.sum(cm):.2%} "
          f"({np.sum(cm.diagonal())}/{np.sum(cm)})")

In [ ]:
import time

print("Inference Benchmark")
print("=" * 60)

sample_batch = next(iter(test_generator))[0]

for name, mod in [('MobileNetV2', model_mobilenet), ('EfficientNetB0', model_efficientnet)]:
    mod.predict(sample_batch, verbose=0)  # warmup
    n_runs = 100
    start = time.perf_counter()
    for _ in range(n_runs):
        mod.predict(sample_batch, verbose=0)
    elapsed = (time.perf_counter() - start) / n_runs
    ms_per_img = elapsed / sample_batch.shape[0] * 1000
    print(f"{name:20s}  {elapsed*1000:>8.2f} ms/batch  |  {ms_per_img:>7.2f} ms/image")

In [ ]:
import matplotlib.pyplot as plt

print("Sample Predictions")
print("=" * 60)

test_generator.reset()
images, labels = next(iter(test_generator))
class_labels = list(test_generator.class_indices.keys())

for name, mod in [('MobileNetV2', model_mobilenet), ('EfficientNetB0', model_efficientnet)]:
    preds = mod.predict(images, verbose=0)
    pred_classes = np.argmax(preds, axis=1)
    true_classes = np.argmax(labels, axis=1)

    fig, axes = plt.subplots(4, 8, figsize=(20, 10))
    fig.suptitle(f'{name} - Sample Predictions (green=correct, red=wrong)', fontsize=14)
    for i, ax in enumerate(axes.flat):
        if i >= len(images):
            ax.axis('off')
            continue
        img = images[i]
        true = class_labels[true_classes[i]]
        pred = class_labels[pred_classes[i]]
        correct = true_classes[i] == pred_classes[i]
        ax.imshow(img)
        ax.set_title(f'{pred}', fontsize=8, color='green' if correct else 'red')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    print(f"{name}: {np.sum(true_classes == pred_classes)}/{len(images)} correct in this batch")

In [ ]:
import tensorflow as tf
import json, time, numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from datetime import datetime

results = {
    'dataset': 'Fruits-360 (13-class + Background)',
    'timestamp': datetime.now().isoformat(),
    'image_size': list(IMG_SIZE),
    'batch_size': BATCH_SIZE,
    'classes': final_classes,
    'models': {}
}

test_generator.reset()
y_true = test_generator.classes
class_labels = list(test_generator.class_indices.keys())

for short_name, display_name, mod, history_list in [
    ('mobilenetv2', 'MobileNetV2', model_mobilenet,
     [('phase1', history_mobilenet), ('finetune', history_mobilenet_fine)]),
    ('efficientnetb0', 'EfficientNetB0', model_efficientnet,
     [('phase1', history_efficientnet), ('finetune', history_efficientnet_fine)]),
]:
    print(f"\n{'='*60}\n{display_name}\n{'='*60}")

    # ---- Test evaluation ----
    test_generator.reset()
    test_loss, test_acc, test_top3 = mod.evaluate(test_generator, verbose=1)
    print(f"Test Accuracy: {test_acc:.4f} | Top-3: {test_top3:.4f} | Loss: {test_loss:.4f}")

    # ---- Per-class metrics ----
    test_generator.reset()
    y_pred_prob = mod.predict(test_generator, verbose=1)
    y_pred = np.argmax(y_pred_prob, axis=1)

    cr = classification_report(y_true, y_pred, target_names=class_labels, output_dict=True, zero_division=0)
    cm = confusion_matrix(y_true, y_pred).tolist()
    class_acc = (cm.diagonal() if isinstance(cm, np.ndarray) else
                 [cm[i][i] / sum(row) for i, row in enumerate(cm)])
    if isinstance(class_acc, np.ndarray):
        class_acc = class_acc.tolist()
    else:
        class_acc = [round(a, 4) for a in class_acc]

    print(f"\nClassification Report:\n{classification_report(y_true, y_pred, target_names=class_labels)}")
    for i, label in enumerate(class_labels):
        acc = cm[i][i] / sum(cm[i]) if sum(cm[i]) > 0 else 0
        print(f"  {label:20s}: {acc:.2%} ({cm[i][i]}/{sum(cm[i])})")
    overall = sum(cm[i][i] for i in range(len(cm))) / sum(sum(row) for row in cm)
    print(f"  {'Overall':20s}: {overall:.2%} ({sum(cm[i][i] for i in range(len(cm)))}/{sum(sum(row) for row in cm)})")

    # ---- Training history ----
    history_dict = {}
    for phase_name, hist in history_list:
        history_dict[phase_name] = {
            k: [float(v) for v in vals[-5:]]  # last 5 epochs
            for k, vals in hist.history.items()
        }

    # ---- Inference benchmark ----
    sample_batch = next(iter(test_generator))[0]
    mod.predict(sample_batch, verbose=0)  # warmup
    n_runs = 50
    start = time.perf_counter()
    for _ in range(n_runs):
        mod.predict(sample_batch, verbose=0)
    elapsed_ms = (time.perf_counter() - start) / n_runs * 1000
    ms_per_img = elapsed_ms / sample_batch.shape[0]

    # ---- Assemble model results ----
    results['models'][short_name] = {
        'display_name': display_name,
        'test_loss': round(float(test_loss), 4),
        'test_accuracy': round(float(test_acc), 4),
        'test_top3_accuracy': round(float(test_top3), 4),
        'classification_report': cr,
        'confusion_matrix': cm,
        'class_labels': class_labels,
        'overall_accuracy': round(overall, 4),
        'training_history': history_dict,
        'inference_ms_per_batch': round(elapsed_ms, 2),
        'inference_ms_per_image': round(ms_per_img, 2)
    }

# ---- Save results ----
results_path = '/content/drive/MyDrive/CropIQ/results.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to {results_path}")

# ---- Save models & TFLite ----
for name, mod in [('mobilenet', model_mobilenet), ('efficientnet', model_efficientnet)]:
    mod.save(f'/content/drive/MyDrive/CropIQ/final_{name}.keras')
    converter = tf.lite.TFLiteConverter.from_keras_model(mod)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter.convert()
    with open(f'/content/drive/MyDrive/CropIQ/{name}_model.tflite', 'wb') as f:
        f.write(tflite_model)

labels = list(train_generator.class_indices.keys())
with open('/content/drive/MyDrive/CropIQ/labels.txt', 'w') as f:
    for lbl in labels:
        f.write(lbl + '\n')

print("Done! Files saved in Drive (CropIQ folder):")
print("- results.json")
print("- final_mobilenet.keras / mobilenet_model.tflite")
print("- final_efficientnet.keras / efficientnet_model.tflite")
print("- labels.txt")
print("Label order:", labels)